# 03. Biến Đổi Dữ Liệu, Thực Hiện Thuật Toán và Đánh Giá So Sánh
**Học phần:** Khai thác dữ liệu - Nhóm 12  
---
### Đối chiếu với Mục 1.2 - Pipeline bắt buộc của đồ án:
- [x] **Bước 4 (Phần 2):** Chọn thuộc tính (chỉ lấy 6 yếu tố cơ sở, loại bỏ target).
- [x] **Bước 6:** Biến đổi dữ liệu sang biểu diễn phù hợp với thuật toán (ma trận đặc trưng số đã scale).
- [x] **Bước 7:** Thực hiện thuật toán và lựa chọn tham số (khảo sát Elbow & Silhouette, cấu hình tham số có cơ sở khoa học).
- [x] **Bước 8:** Đánh giá và so sánh ít nhất hai phương án/cấu hình (K-Means vs Hierarchical vs DBSCAN).

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.features.feature_engineering import scale_features, FEATURE_COLUMNS
from src.models.clustering import find_optimal_k_kmeans, run_kmeans, run_hierarchical, run_dbscan, evaluate_clustering
from src.visualization.plots import plot_elbow_silhouette

figures_dir = project_root / 'reports' / 'figures'

## [Bước 6/11] Biến đổi dữ liệu sang biểu diễn phù hợp với thuật toán gom cụm
Vì thuật toán K-Means và Phân cấp dựa trên khoảng cách Euclidean, sự chênh lệch thang đo giữa các biến sẽ làm sai lệch trọng số khoảng cách. Ta thực hiện chuẩn hóa Z-score (`StandardScaler`).

In [ ]:
df = pd.read_csv(project_root / 'data' / 'interim' / 'happiness_merged.csv')
# Chọn năm 2019 làm tập thực nghiệm chính (năm mới nhất và đầy đủ chỉ số)
df_2019 = df[df['year'] == 2019].dropna(subset=FEATURE_COLUMNS).copy().reset_index(drop=True)
print(f'Tập dữ liệu thực nghiệm năm 2019: {df_2019.shape[0]} quốc gia')

X_scaled, scaler = scale_features(df_2019, method='standard')
print('Kích thước ma trận đặc trưng chuẩn hóa X_scaled:', X_scaled.shape)

## [Bước 7/11] Thực hiện thuật toán và lựa chọn tham số có cơ sở
Khảo sát số cụm $k \in [2, 9]$ trên K-Means dựa trên hai tiêu chí:
1. **Inertia (WCSS)** - Tìm điểm gập khuỷu tay (Elbow method).
2. **Silhouette Score** - Đo độ gắn kết nội cụm và tách biệt giữa các cụm.

In [ ]:
metrics_df = find_optimal_k_kmeans(X_scaled, k_range=range(2, 10))
display(metrics_df)
plot_elbow_silhouette(metrics_df, save_path=str(figures_dir / 'elbow_silhouette_kmeans.png'))
plt.show()

In [ ]:
# Huấn luyện K-Means với k tối ưu được phân tích ở trên (ví dụ k = 3)
optimal_k = 3
kmeans_labels, kmeans_model = run_kmeans(X_scaled, n_clusters=optimal_k)
df_2019['cluster_kmeans'] = kmeans_labels
print(f'Phân bố số lượng quốc gia ở mỗi cụm K-Means (k={optimal_k}):')
print(df_2019['cluster_kmeans'].value_counts())

## [Bước 8/11] Đánh giá và so sánh ít nhất hai phương án gom cụm
So sánh mô hình cơ sở **K-Means** với 2 phương án đối sánh:
1. **Hierarchical Agglomerative Clustering** (Linkage = 'ward').
2. **DBSCAN** (Gom cụm dựa trên mật độ và phát hiện nhiễu).

In [ ]:
# Phương án đối sánh 1: Hierarchical Ward
hier_labels, hier_model = run_hierarchical(X_scaled, n_clusters=optimal_k, linkage='ward')
df_2019['cluster_hierarchical'] = hier_labels

# Phương án đối sánh 2: DBSCAN
dbscan_labels, dbscan_model = run_dbscan(X_scaled, eps=1.2, min_samples=4)
df_2019['cluster_dbscan'] = dbscan_labels

In [ ]:
# Bảng so sánh tổng hợp các chỉ số nội tại
comparison_records = [
    {'Phương pháp': f'K-Means (k={optimal_k})', **evaluate_clustering(X_scaled, kmeans_labels)},
     'Ưu điểm chính': 'Hội tụ nhanh, cụm cân đối', 'Nhược điểm': 'Nhạy cảm khởi tạo tâm'},
    {'Phương pháp': f'Hierarchical Ward (k={optimal_k})', **evaluate_clustering(X_scaled, hier_labels)},
     'Ưu điểm chính': 'Cấu trúc dạng cây (Dendrogram)', 'Nhược điểm': 'Độ phức tạp O(n^2)'},
    {'Phương pháp': 'DBSCAN', **evaluate_clustering(X_scaled, dbscan_labels)},
     'Ưu điểm chính': 'Phát hiện nhiễu và cụm phi cầu', 'Nhược điểm': 'Khó chọn eps/min_samples'}
]
comparison_df = pd.DataFrame(comparison_records)
display(comparison_df)
comparison_df.to_csv(figures_dir / 'model_comparison_metrics.csv', index=False)